# Preprocesamiento y Preparación de Datos

**Caso de Estudio:** Spotify Tracks Dataset  
**Asignatura:** MLY1101 — Machine Learning  
**Evaluación Parcial N°1**  
**Por:** Felipe Ahumada Silva y Francisca Carrasco Lozano

---

## Descripción

Este notebook corresponde a la **Fase 3 (Preparación de datos)** de la metodología CRISP-DM
aplicada al caso de estudio Spotify Tracks Dataset. Toma como punto de partida la limpieza y
los hallazgos documentados en [`analisis_exploratorio.ipynb`](analisis_exploratorio.ipynb) (duplicados por track_id, duplicados de audio bajo
track_id distinto, rangos inválidos, outliers, correlaciones) y construye, mediante Pipeline/ColumnTransformer de
scikit-learn, el flujo de transformación que deja los datos listos para el proceso de
modelamiento (target: **popularity**).

**Alcance:** este notebook cubre únicamente la preparación y transformación de datos. El
entrenamiento y la evaluación de modelos predictivos quedan fuera de su alcance.

Mantiene el uso de pipeline como buena práctica de la industria.

---

## Requisitos de Software

Este notebook fue desarrollado con Python 3.12. Bibliotecas necesarias:

- pandas (>=1.1.0)
- numpy (>=2.0.0)
- scikit-learn (>=1.3)

# Importación de librerías

In [1]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

# Reconstrucción del dataset limpio

La consolidación de canciones repetidas por track_id y la eliminación del registro con
duration_ms = 0 ya fueron exploradas y justificadas en detalle en
[`analisis_exploratorio.ipynb`](analisis_exploratorio.ipynb); aquí solo se reproducen para
que este notebook sea autocontenido y ejecutable de forma independiente.

El EDA documentó además, sin resolverlo, un tercer tipo de duplicado: audio idéntico bajo
track_id distintos (misma canción en distintos releases/álbumes, ~9.284 filas en 3.022
grupos). Esa decisión se toma acá, porque corresponde a la etapa de preparación de datos y
no a la exploración.

In [2]:
df = pd.read_csv('../data/Spotify_Tracks_Dataset.csv', index_col=0)

# Consolidación: una fila por track_id único (ver justificación en el EDA)
agregaciones = {col: 'first' for col in df.columns if col not in ['track_id', 'track_genre', 'popularity']}
agregaciones['track_genre'] = lambda generos: ';'.join(sorted(set(generos)))
agregaciones['popularity'] = 'mean'

df = df.groupby('track_id', as_index=False).agg(agregaciones)
df['popularity'] = df['popularity'].round().astype(int)

# Registro con duration_ms = 0 y metadatos nulos: se descarta por ser un error de carga (ver EDA)
df = df[df['duration_ms'] > 0].reset_index(drop=True)

print(f'Filas: {len(df)} | Columnas: {df.shape[1]}')
df.head()

Filas: 89740 | Columnas: 20


,track_id,artists,album_name,track_name,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,popularity
0,0000vdREvCVMxbQTkS888c,Rill,Lolly,Lolly,160725,True,0.910,0.374,8,-9.844,0,0.1990,0.075700,0.00301,0.1540,0.432,104.042,4,german,44
1,000CC8EParg64OmTxVnZ0p,Glee Cast,Glee Love Songs,It's All Coming Back To Me Now (Glee Cast Vers...,322933,False,0.269,0.516,0,-7.361,1,0.0366,0.406000,0.00000,0.1170,0.341,178.174,4,club,47
2,000Iz0K615UepwSJ5z2RE5,Paul Kalkbrenner;Pig&Dan,X,Böxig Leise - Pig & Dan Remix,515360,False,0.686,0.560,5,-13.264,0,0.0462,0.001140,0.18100,0.1110,0.108,119.997,4,minimal-techno,22
3,000RDCYioLteXcutOjeweY,Jordan Sandhu,Teeje Week,Teeje Week,190203,False,0.679,0.770,0,-3.537,1,0.1900,0.058300,0.00000,0.0825,0.839,161.721,4,hip-hop,62
4,000qpdoc97IMTBvF8gwcpy,Paul Kalkbrenner,Zeit,Tief,331240,False,0.519,0.431,6,-13.606,0,0.0291,0.000964,0.72000,0.0916,0.234,129.971,4,minimal-techno,19


### Consolidación adicional: audio idéntico bajo track_id distinto

Se colapsa cada grupo de audio duplicado (mismo fingerprint de 13 columnas de audio, ver
EDA) a una sola fila, promediando popularity entre los releases, la misma convención ya
usada arriba para los duplicados por track_id ,en vez de quedarse con la fila de mayor
popularidad. Elegir el máximo introduciría un sesgo optimista sistemático en el target (cada
canción con reediciones quedaría con su mejor versión posible, no con una medida
representativa de "cuánto se escucha esta canción"), mientras que promediar reutiliza el
mismo criterio ya aplicado al duplicado por género, sin necesidad de una regla nueva.

In [3]:
audio_cols = ['duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode',
              'speechiness', 'acousticness', 'instrumentalness', 'liveness',
              'valence', 'tempo', 'time_signature']

filas_antes_audio = len(df)

agregaciones_audio = {
    col: 'first' for col in df.columns if col not in audio_cols + ['track_genre', 'popularity']
}
agregaciones_audio['track_genre'] = lambda generos: ';'.join(sorted(set(generos)))
agregaciones_audio['popularity'] = 'mean'

df = df.groupby(audio_cols, as_index=False).agg(agregaciones_audio)
df['popularity'] = df['popularity'].round().astype(int)

print(f'Filas antes de consolidar por audio: {filas_antes_audio}')
print(f'Filas después: {len(df)} (-{filas_antes_audio - len(df)})')
df.head()

Filas antes de consolidar por audio: 89740
Filas después: 83478 (-6262)


,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_id,artists,album_name,track_name,explicit,track_genre,popularity
0,8586,0.000,0.0400,8,-29.714,0,0.0000,0.928,0.956,0.115,0.000,0.000,0,6hsyfegVY5yklJneM40mWi,Leila Bela,Angra Manyu,The Exorsism Begins...,False,iranian,0
1,13386,0.000,0.2240,11,-22.196,1,0.0000,0.970,0.000,0.907,0.000,0.000,0,38Ogh3rsHba83kXx13gbKs,Leila Bela,Angra Manyu,V-4,False,iranian,0
2,15800,0.251,0.5080,5,-10.564,0,0.3160,0.969,0.999,0.952,0.000,184.051,3,1HVjSh7scH1PaPiLjy2LEu,Leila Bela;Leila's Opera Class,Angra Manyu,Screams for a Finale! (feat. Leila's Opera Class),False,iranian,0
3,17453,0.467,0.0301,2,-28.518,0,0.0428,0.995,0.900,0.124,0.000,84.375,4,5YKCM3jbJ8lqUXUwfU7KwZ,Wolfgang Amadeus Mozart;Ingrid Haebler,Mozart: The Complete Piano Sonatas,"Andante in C Major, K. 1a",False,classical,0
4,17826,0.372,0.2780,8,-16.882,1,0.0370,0.985,0.921,0.164,0.912,89.032,1,1T5QvLF9lO4HO3OZQbaX9p,Robert Schumann;Pavel Nersessian,"Schumann, Poulenc & Others: Piano Works (Live ...","Carnaval, Op. 9: No. 20, Pause (Live in Japan,...",False,classical;german,0


# Fase 3 — Preparación de datos

El resto del notebook desarrolla la preparación en cinco pasos:

1. Tratamiento de los valores inválidos detectados en el EDA (tempo y time_signature en 0).
2. Definición de variables (target, features) y partición train/test.
3. Transformadores personalizados reutilizables (outliers, multicolinealidad, formato de salida).
4. Ensamblado del pipeline de preparación (ColumnTransformer + Pipeline).
5. Ajuste sobre el conjunto de entrenamiento y verificación del resultado.

## 1. Tratamiento de valores inválidos detectados en el EDA

El EDA confirmó dos fallas puntuales del algoritmo de estimación de Spotify, no variabilidad
real de los datos. Los conteos son posteriores a la consolidación por audio de la sección
anterior, que ya fusionó algunas filas que compartían el mismo valor inválido entre releases:

- **time_signature = 0** (129 filas): 96% de estas filas también tiene tempo = 0 y el 81%
  pertenece al género sleep (sonidos ambientales sin pulso musical). No es un compás real.
- **tempo = 0** (124 filas, subconjunto de las anteriores): ninguna canción tiene 0 BPM, es
  físicamente imposible.

En lugar de eliminar estas filas (perdiendo el resto de sus atributos, que sí son válidos), se
marcan como valores faltantes (NaN) en su propia columna, delegando su tratamiento al
SimpleImputer del pipeline más abajo. time_signature = 1 no se toca: el EDA confirmó que
representa compases reales, poco frecuentes pero válidos.

In [4]:
mask_time_signature_invalido = df['time_signature'] == 0
mask_tempo_invalido = df['tempo'] == 0

print(f"time_signature = 0 (inválido): {mask_time_signature_invalido.sum()} filas")
print(f"tempo = 0 (inválido): {mask_tempo_invalido.sum()} filas")

df.loc[mask_time_signature_invalido, 'time_signature'] = np.nan
df.loc[mask_tempo_invalido, 'tempo'] = np.nan

print(f"Nulos tras la marcación: {df[['time_signature', 'tempo']].isna().sum().to_dict()}")

time_signature = 0 (inválido): 129 filas
tempo = 0 (inválido): 124 filas
Nulos tras la marcación: {'time_signature': 129, 'tempo': 124}


## Ingeniería de características adicionales

Se agregan dos variables derivadas de los atributos de audio ya existentes.:

- **distancia_duracion_optima** y,
- **es_calmado_positivo**

Se explica en [`analisis_exploratorio.ipynb`](analisis_exploratorio.ipynb) más a detalle el por qué de ellas.

In [5]:
df['distancia_duracion_optima'] = (df['duration_ms'] / 60000 - 3.75).abs()
df['es_calmado_positivo'] = ((df['valence'] >= 0.5) & (df['energy'] < 0.5)).astype(int)

df[['distancia_duracion_optima', 'es_calmado_positivo']].describe()

,distancia_duracion_optima,es_calmado_positivo
count,83478.000000,83478.000000
mean,1.104448,0.079973
std,1.580197,0.271253
min,0.000000,0.000000
25%,0.371550,0.000000
50%,0.791283,0.000000
75%,1.403333,0.000000
max,83.538250,1.000000


## 2. Definición de variables y partición train/test

**Target:** popularity (variable continua 0-100), se plantea como un problema de regresión,
consistente con el objetivo de negocio del EDA (comprender qué atributos se asocian a la
popularidad de una canción).

**Columnas excluidas de las features:**

- track_id, artists, album_name, track_name: identificadores de altísima
  cardinalidad (73.261 nombres de canción y 45.880 álbumes distintos sobre 83.478 filas, tras
  la consolidación por audio de este notebook (ver sección anterior)), donde casi cada valor aparece una sola vez. No hay forma de que un modelo generalice
  a partir de una categoría con un ejemplo, y transformarlas en una señal útil (largo/palabras
  del título, fama del artista vía el álbum) requeriría ingeniería de variables adicional que
  queda fuera del alcance de esta preparación.

**track_genre se decidió tratar de una forma especial**.

Se evaluaron tres formas de incorporarla:

- **Multi-hot completo** (MultiLabelBinarizer, ~114 columnas binarias, una por género):
  no pierde información ni requiere ver el target, pero infla la tabla final de forma
  considerable (de ~26 a más de 140 columnas) y no agrupa géneros afines entre sí.
- **Agrupación en familias de género** (mapear los 114 géneros a ~10-15 familias musicales y
  hacer one-hot sobre esas familias): más compacto e interpretable, pero exige construir el
  mapeo a mano, con un criterio necesariamente subjetivo.
- **Codificación por promedio de género** (*mean/target encoding*, la elegida): comprime toda
  la señal en una sola columna numérica —el promedio de popularidad histórica del/los género(s)
  de la canción, calculado exclusivamente sobre el conjunto de entrenamiento para no filtrar
  información del conjunto de prueba hacia el modelo (data leakage). Es la opción más compacta
  de las tres y la más directa de justificar con el hallazgo anterior, a costa de perder el
  detalle de qué género específico es responsable del efecto.

**Features numéricas** (mediciones continuas de audio): duration_ms, danceability,
energy, loudness, speechiness, acousticness, instrumentalness, liveness, valence,
tempo, distancia_duracion_optima.

**Features categóricas** (códigos nominales/discretos, no continuos): key (clase de tono,
0-11), mode (mayor/menor), time_signature (compás), explicit y es_calmado_positivo.

distancia_duracion_optima y es_calmado_positivo son variables derivadas de los atributos de
audio ya existentes, definidas en la sección "Ingeniería de características adicionales" más
arriba.

**Feature de género:** track_genre, tratada aparte con el transformador GenreMeanEncoder
definido en la siguiente sección, ya que su codificación necesita ver el target durante el
ajuste (a diferencia de todas las demás).

La partición train/test se realiza antes de ajustar cualquier transformador, para que las
estadísticas de imputación, recorte de atípicos, escalado y el promedio de popularidad por
género se aprendan solo del conjunto de entrenamiento y no se filtre información del conjunto de
prueba (data leakage).

In [6]:
target = 'popularity'

features_num = [
    'duration_ms', 'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
    'distancia_duracion_optima'
]
features_cat = ['key', 'mode', 'time_signature', 'explicit', 'es_calmado_positivo']
feature_genero = ['track_genre']

X = df[features_num + features_cat + feature_genero]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Entrenamiento: {X_train.shape} | Prueba: {X_test.shape}')

Entrenamiento: (66782, 17) | Prueba: (16696, 17)


## 3. Transformadores personalizados

Se definen tres transformadores compatibles con la API de scikit-learn
(BaseEstimator, TransformerMixin), pensados para poder combinarse dentro de un Pipeline:

- **Winsorizer**: recorta (clip) los percentiles extremos de cada columna numérica. Se
  justifica en los boxplots del EDA, que mostraron outliers marcados en duration_ms
  (canciones de varias horas), loudness (hasta -49 dB) y en speechiness,
  instrumentalness y liveness (variables concentradas cerca de 0 con colas largas).
- **CorrelationFilter**: elimina columnas con correlación absoluta por sobre un umbral, como
  resguardo frente a multicolinealidad. La matriz de correlación del EDA no mostró pares por
  sobre 0.9 (el máximo fue energy-loudness con 0.76), por lo que no se espera que descarte
  columnas hoy, pero se mantiene como paso metodológico reutilizable si se agregan más features.
- **DataFrameConverter**: convierte la salida (un array de NumPy) del ColumnTransformer de
  vuelta en un DataFrame con nombres de columna legibles, útil para inspeccionar el resultado
  de la preparación.
- **GenreMeanEncoder**: codifica track_genre como el promedio de popularidad histórica de
  su(s) género(s) (ver sección anterior). Es el único transformador de este notebook que
  necesita el target durante el ajuste (**fit(X, y)**, no **fit(X, y=None)**): "explota" los géneros
  de cada canción del conjunto de entrenamiento, calcula el promedio de popularity por género
  únicamente con esos datos, y en transform asigna a cada canción el promedio de las medias de
  sus propios géneros (si una canción tiene más de uno). Un género no visto en entrenamiento cae
  al promedio global de entrenamiento, de forma análoga a **handle_unknown='ignore'** en
  **OneHotEncoder**.

In [7]:
class Winsorizer(BaseEstimator, TransformerMixin):
    """
    Tratamiento de atípicos vía recorte por percentiles.

    limits acepta una tupla (mismo recorte para todas las columnas, comportamiento
    original) o un diccionario {columna: (inferior, superior)} para darle a cada
    columna su propio recorte. Una columna con (0.0, 0.0) no se recorta (quantile(0) y
    quantile(1) son el mínimo y el máximo de la columna, np.clip no cambia nada).
    """
    def __init__(self, limits=(0.05, 0.05)):
        self.limits = limits

    def _limites_columna(self, col):
        if isinstance(self.limits, dict):
            return self.limits.get(col, (0.0, 0.0))
        return self.limits

    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            self.columns_ = X.columns
        else:
            self.columns_ = np.arange(X.shape[1])
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.columns_).astype('float64')
        for col in self.columns_:
            inferior, superior = self._limites_columna(col)
            if inferior == 0 and superior == 0:
                continue
            lower = X[col].quantile(inferior)
            upper = X[col].quantile(1 - superior)
            X[col] = np.clip(X[col], lower, upper)
        return X

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.array(self.columns_)
        return np.array(input_features)

In [8]:
class CorrelationFilter(BaseEstimator, TransformerMixin):
    """
    Elimina variables con alta correlación (multicolinealidad).
    """
    def __init__(self, threshold=0.9):
        self.threshold = threshold
        self.columns_to_drop_ = None

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        corr_matrix = X_df.corr().abs()
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        self.columns_to_drop_ = [
            col for col in upper.columns if any(upper[col] > self.threshold)
        ]
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X)
        return X_df.drop(columns=self.columns_to_drop_, errors='ignore').values

In [9]:
class DataFrameConverter(BaseEstimator, TransformerMixin):
    """
    Convierte el array de ColumnTransformer en DataFrame con nombres de columnas.
    """
    def __init__(self, preprocessor):
        self.preprocessor = preprocessor
        self.feature_names_ = None

    def fit(self, X, y=None):
        self.feature_names_ = self.preprocessor.get_feature_names_out()
        return self

    def transform(self, X):
        return pd.DataFrame(X, columns=self.feature_names_)

In [10]:
class GenreMeanEncoder(BaseEstimator, TransformerMixin):
    """
    Codifica track_genre (multietiqueta, géneros separados por ';') como el promedio de
    popularidad histórica de sus géneros. Para una canción con más de un género se promedian
    las medias de cada uno. Las medias se calculan exclusivamente sobre los datos vistos en
    fit (el conjunto de entrenamiento), para no filtrar información del target de prueba.
    """
    def __init__(self, column='track_genre'):
        self.column = column

    def fit(self, X, y):
        generos = X[self.column].str.split(';').explode()
        y_generos = y.loc[generos.index]
        self.medias_genero_ = y_generos.groupby(generos.values).mean()
        self.media_global_ = y.mean()
        return self

    def transform(self, X):
        def promedio_generos(generos_cancion):
            return np.mean([
                self.medias_genero_.get(genero, self.media_global_)
                for genero in generos_cancion
            ])
        columna = X[self.column].str.split(';').apply(promedio_generos)
        return columna.to_frame(name='genero_popularidad_media')

    def get_feature_names_out(self, input_features=None):
        return np.array(['genero_popularidad_media'])

## 4. Ensamblado del pipeline de preparación

**Pipeline numérico:** Winsorizer → SimpleImputer(strategy='mean') → StandardScaler.
El recorte de atípicos se aplica antes de imputar: Winsorizer calcula los percentiles
ignorando los NaN recién introducidos en tempo y los deja intactos al recortar, por lo que
el SimpleImputer sigue siendo el único paso que los completa (con la media de la columna, ya
calculada sobre el resto de los valores).

**Pipeline categórico:** SimpleImputer(strategy='most_frequent') → `OneHotEncoder`. La moda
también resuelve, de forma natural, los NaN introducidos en time_signature (el compás más
frecuente del dataset es 4/4). drop='first' evita la trampa de las variables dummy y
handle_unknown='ignore' deja el pipeline seguro ante una categoría no vista en producción.

**Pipeline de género:** GenreMeanEncoder → StandardScaler, para que la nueva columna quede
en una escala comparable al resto de las features numéricas.

### Recorte de Winsorizer por columna, no uniforme

La justificación de más arriba explica por qué se usa Winsorizer en general, pero un mismo recorte (5% inferior + 5% superior) para las 11 columnas numéricas trata igual a columnas sin outliers reales que a columnas con outliers grandes y concentrados. Se mide el % real de valores atípicos de cada columna (regla IQR, 1.5× el rango intercuartílico) y hacia qué lado están, sobre el mismo `df` ya consolidado que alimenta el pipeline (esto es una decisión de diseño fija, igual que el umbral 0.9 de `CorrelationFilter`, no una estadística aprendida de datos de prueba: los valores de recorte en sí los sigue fijando `fit` únicamente con `X_train`, acá solo se decide cuánto recortar en cada caso):

In [11]:
print(f"{'columna':22s} {'% abajo':>9s} {'% arriba':>9s}")
for col in features_num:
    q1, q3 = df[col].quantile([.25, .75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    print(f"{col:22s} {(df[col] < lo).mean()*100:8.2f}% {(df[col] > hi).mean()*100:8.2f}%")

columna                  % abajo  % arriba
duration_ms                0.10%     4.59%
danceability               0.49%     0.00%
energy                     0.00%     0.00%
loudness                   5.57%     0.00%
speechiness                0.00%    11.91%
acousticness               0.00%     0.00%
instrumentalness           0.00%    21.53%
liveness                   0.00%     7.94%
valence                    0.00%     0.00%
tempo                      0.02%     0.37%
distancia_duracion_optima     0.00%     5.21%


`energy`, `acousticness` y `valence` no tienen ningún outlier real (0% en ambos lados): recortarlas igual que al resto solo descarta datos válidos de los extremos sin motivo. En el otro extremo, `instrumentalness` (21,5%) y `speechiness` (11,9%) concentran outliers reales muy por sobre el 5% que se les aplicaba antes. Los límites de abajo usan el % real medido, con un tope de 10% por lado: recortar más que eso metería a más de 1 de cada 10 canciones en el mismo valor de borde, generando un pico artificial en la distribución en vez de tratar un outlier puntual.

In [12]:
limits_winsorizer = {
    'duration_ms':               (0.001, 0.045),
    'danceability':              (0.005, 0.000),
    'energy':                    (0.000, 0.000),
    'loudness':                  (0.055, 0.000),
    'speechiness':               (0.000, 0.100),   # tope: real es 11.9%
    'acousticness':              (0.000, 0.000),
    'instrumentalness':          (0.000, 0.100),   # tope: real es 21.5%
    'liveness':                  (0.000, 0.080),
    'valence':                   (0.000, 0.000),
    'tempo':                     (0.000, 0.004),
    'distancia_duracion_optima': (0.000, 0.050),
}

In [13]:
numeric_transformer = Pipeline(steps=[
    ('winsorizer', Winsorizer(limits_winsorizer)),
    ('imputer',    SimpleImputer(strategy='mean')),
    ('scaler',     StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(drop='first', handle_unknown='ignore'))
])

genero_transformer = Pipeline(steps=[
    ('genero_encoder', GenreMeanEncoder()),
    ('scaler',         StandardScaler())
])

preprocesador = ColumnTransformer(
    transformers=[
        ('num',    numeric_transformer,    features_num),
        ('cat',    categorical_transformer, features_cat),
        ('genero', genero_transformer,     feature_genero)
    ],
    remainder='drop',
    force_int_remainder_cols=False
)

In [14]:
pipeline_preparacion = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('conversion',    DataFrameConverter(preprocesador)),
    ('colinealidad',  CorrelationFilter(threshold=0.9))
])

pipeline_preparacion

Pipeline(steps=[('preprocesador',
                 ColumnTransformer(force_int_remainder_cols=False,
                                   transformers=[('num',
                                                  Pipeline(steps=[('winsorizer',
                                                                   Winsorizer(limits={'acousticness': (0.0,
                                                                                                       0.0),
                                                                                      'danceability': (0.005,
                                                                                                       0.0),
                                                                                      'distancia_duracion_optima': (0.0,
                                                                                                                    0.05),
                                                                                      'duration_ms': (0.001,
                                                                                                      0.045),
                                                                                      'energy': (0.0,
                                                                                                 0.0),
                                                                                      'instrumentalness': (0.0,
                                                                                                           0.1),
                                                                                      'liveness': (0.0,
                                                                                                   0.08),
                                                                                      'loudness...
                                                                                  Pipeline(steps=[('imputer',
                                                                                                   SimpleImputer(strategy='most_frequent')),
                                                                                                  ('onehot',
                                                                                                   OneHotEncoder(drop='first',
                                                                                                                 handle_unknown='ignore'))]),
                                                                                  ['key',
                                                                                   'mode',
                                                                                   'time_signature',
                                                                                   'explicit',
                                                                                   'es_calmado_positivo']),
                                                                                 ('genero',
                                                                                  Pipeline(steps=[('genero_encoder',
                                                                                                   GenreMeanEncoder()),
                                                                                                  ('scaler',
                                                                                                   StandardScaler())]),
                                                                                  ['track_genre'])]))),
                ('colinealidad', CorrelationFilter())])

## 5. Ajuste sobre el conjunto de entrenamiento

In [15]:
X_train_prep = pipeline_preparacion.fit_transform(X_train, y_train)
X_test_prep = pipeline_preparacion.transform(X_test)

columnas_descartadas = pipeline_preparacion.named_steps['colinealidad'].columns_to_drop_
columnas_finales = [
    col for col in pipeline_preparacion.named_steps['conversion'].feature_names_
    if col not in columnas_descartadas
]

X_train_prep = pd.DataFrame(X_train_prep, columns=columnas_finales)
X_test_prep = pd.DataFrame(X_test_prep, columns=columnas_finales)

print(f"Columnas antes del filtro de colinealidad: {len(pipeline_preparacion.named_steps['conversion'].feature_names_)}")
print(f'Columnas descartadas por colinealidad (umbral 0.9): {columnas_descartadas}')
print(f'Forma final -> Entrenamiento: {X_train_prep.shape} | Prueba: {X_test_prep.shape}')

X_train_prep.head()

Columnas antes del filtro de colinealidad: 29
Columnas descartadas por colinealidad (umbral 0.9): []
Forma final -> Entrenamiento: (66782, 29) | Prueba: (16696, 29)


,num__duration_ms,num__danceability,num__energy,num__loudness,num__speechiness,num__acousticness,num__instrumentalness,num__liveness,num__valence,num__tempo,...,cat__key_9.0,cat__key_10.0,cat__key_11.0,cat__mode_1.0,cat__time_signature_3.0,cat__time_signature_4.0,cat__time_signature_5.0,cat__explicit_1.0,cat__es_calmado_positivo_1.0,genero__genero_popularidad_media
0,-0.272367,0.099455,0.272780,0.193871,-0.615620,-0.925609,-0.556639,-0.500380,-0.025073,0.255628,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.519628
1,0.576781,-0.501902,1.004042,1.106610,-0.499262,-0.965130,1.402501,-0.743119,-0.218870,-0.927066,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,-0.627098
2,0.594680,0.695140,0.396591,0.303644,0.901041,0.991009,-0.556647,-0.863815,0.628516,-0.926728,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,-0.281238
3,-0.211082,-0.632386,-2.231308,-2.398621,-0.768088,0.870356,2.149740,-0.473409,-0.177071,-0.892793,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.173700
4,0.894160,-1.001143,0.868623,0.891060,-0.340775,-0.960539,-0.556358,-0.365525,-0.500065,1.088932,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,-0.665903


In [16]:
print(f'Valores nulos en X_train_prep: {X_train_prep.isna().sum().sum()}')
print(f'Valores nulos en X_test_prep: {X_test_prep.isna().sum().sum()}')

X_train_prep.describe()

Valores nulos en X_train_prep: 0
Valores nulos en X_test_prep: 0


,num__duration_ms,num__danceability,num__energy,num__loudness,num__speechiness,num__acousticness,num__instrumentalness,num__liveness,num__valence,num__tempo,...,cat__key_9.0,cat__key_10.0,cat__key_11.0,cat__mode_1.0,cat__time_signature_3.0,cat__time_signature_4.0,cat__time_signature_5.0,cat__explicit_1.0,cat__es_calmado_positivo_1.0,genero__genero_popularidad_media
count,6.678200e+04,6.678200e+04,6.678200e+04,6.678200e+04,6.678200e+04,6.678200e+04,6.678200e+04,6.678200e+04,6.678200e+04,6.678200e+04,...,66782.000000,66782.000000,66782.000000,66782.000000,66782.000000,66782.000000,66782.000000,66782.000000,66782.000000,6.678200e+04
mean,-1.928984e-16,6.333834e-16,9.253909e-17,8.596905e-17,-2.672701e-16,-6.341282e-17,4.787880e-17,-9.660879e-17,3.141382e-17,-2.691853e-17,...,0.100401,0.066126,0.079033,0.634018,0.084708,0.888039,0.017774,0.084888,0.079408,1.055329e-16
std,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,...,0.300537,0.248503,0.269793,0.481708,0.278449,0.315321,0.132131,0.278717,0.270376,1.000007e+00
min,-2.454234e+00,-2.686700e+00,-2.458812e+00,-2.398621e+00,-1.426110e+00,-9.688674e-01,-5.566469e-01,-1.363453e+00,-1.765444e+00,-3.116728e+00,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-2.773246e+00
25%,-6.606773e-01,-6.380587e-01,-6.983668e-01,-5.167167e-01,-7.038909e-01,-9.223718e-01,-5.566469e-01,-6.992914e-01,-8.420595e-01,-7.628068e-01,...,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,-6.829244e-01
50%,-1.292523e-01,7.676256e-02,1.644448e-01,2.335960e-01,-4.430894e-01,-4.097434e-01,-5.563911e-01,-4.666663e-01,-5.547275e-02,-9.870812e-03,...,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.138199e-01
75%,5.339849e-01,7.348520e-01,8.570155e-01,7.333344e-01,3.092227e-01,8.791844e-01,-1.202341e-01,5.447474e-01,8.109126e-01,6.011223e-01,...,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,7.057497e-01
max,2.341218e+00,2.414115e+00,1.410298e+00,3.003547e+00,2.285296e+00,1.962119e+00,2.149740e+00,2.392263e+00,2.007892e+00,2.627278e+00,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.102283e+00


# Resumen

- Se reconstruyó el dataset consolidado del EDA de forma autocontenida en este notebook, y
  se agregó una consolidación adicional que el EDA dejó documentada pero sin resolver: audio
  idéntico bajo track_id distintos (~9.284 filas en 3.022 grupos), colapsado a una fila por
  grupo promediando popularity entre releases. El dataset final queda en 83.478 canciones
  únicas (89.740 tras la consolidación por track_id, menos 6.262 filas por la consolidación
  por audio).
- Se marcaron como NaN los 129/124 valores de time_signature/tempo identificados como fallas
  de estimación en el EDA (conteo ya sobre el dataset consolidado por audio).
- Se agregaron dos variables de ingeniería de características derivadas de los atributos de
  audio ya existentes: distancia_duracion_optima y es_calmado_positivo.
- Se definieron el target (popularity) y las features numéricas/categóricas para el
  modelamiento. track_genre se incorporó mediante GenreMeanEncoder (promedio de
  popularidad histórica por género, calculado solo con el conjunto de entrenamiento) en vez de
  descartarla, por mostrar una relación con popularity más fuerte que cualquier variable de
  audio.
- La consolidación por audio también mejoró la señal de género: la correlación de GenreMeanEncoder
  con popularity en test subió de 0.5678 (dataset sin esta consolidación) a 0.6182, consistente
  con que se eliminó ruido de etiqueta entre releases del mismo audio.
- Se construyó un pipeline de preparación (Winsorizer, imputación, escalado, codificación
  one-hot, codificación de género y filtro de colinealidad) ajustado exclusivamente sobre el
  conjunto de entrenamiento, para evitar fuga de información hacia el conjunto de prueba.
- **Nota ética/privacidad:** las únicas variables descriptivas de personas (artists) son
  nombres de artistas públicos asociados a un catálogo musical comercial, no datos personales
  de individuos privados; de todas formas se excluyen de las features por no aportar valor
  predictivo, no por motivos de privacidad.

El resultado (X_train_prep, X_test_prep, y_train, y_test) queda listo para la etapa de
modelamiento, que corresponde a un notebook posterior y está fuera del alcance de este
documento.